In [5]:
import hashlib
import numpy as np


# =========================
# Utility: simulate bfloat16
# =========================
def to_bfloat16_sim(x):
    """
    Simulate bfloat16 storage using float32 by truncating the lower 16 bits.
    The returned array is still stored as float32 in NumPy, but its values
    have bfloat16-like precision.
    """
    x = x.astype(np.float32)
    x_uint = x.view(np.uint32)
    x_bf16_uint = x_uint & np.uint32(0xFFFF0000)
    return x_bf16_uint.view(np.float32)


def cast_precision(x, precision):
    if precision == "fp64":
        return x.astype(np.float64)
    elif precision == "fp32":
        return x.astype(np.float32)
    elif precision == "fp16":
        return x.astype(np.float16)
    elif precision == "bf16":
        return to_bfloat16_sim(x)
    else:
        raise ValueError("Unknown precision")


# =========================
# Standard attention
# =========================
def standard_attention(Q, K, V, precision):
    """
    Reference attention:
        O = softmax(QK^T)V
    """

    if precision in ["fp16", "bf16"]:
        # Low-precision inputs, but use fp32 accumulation for numerical stability.
        Qc = Q.astype(np.float32)
        Kc = K.astype(np.float32)
        Vc = V.astype(np.float32)
    else:
        Qc, Kc, Vc = Q, K, V

    S = Qc @ Kc.T

    # Row-wise numerically stable softmax
    S = S - np.max(S, axis=1, keepdims=True)
    P = np.exp(S)
    P = P / np.sum(P, axis=1, keepdims=True)

    O = P @ Vc
    return O


# =========================
# FlashAttention 1 forward
# =========================
def flash_attention_1_forward(Q, K, V, M=256, precision="fp32"):
    """
    FlashAttention 1 forward pass, written in Python-style NumPy code.

    Q, K, V: shape (N, d)
    M: SRAM size measured in number of elements
    """

    N, d = Q.shape

    # FlashAttention paper-style block sizes
    Bc = int(np.ceil(M / (4 * d)))
    Br = min(Bc, d)

    if precision in ["fp16", "bf16"]:
        # Low-precision inputs, but use fp32 accumulation.
        Qc = Q.astype(np.float32)
        Kc = K.astype(np.float32)
        Vc = V.astype(np.float32)
        compute_dtype = np.float32
    else:
        Qc, Kc, Vc = Q, K, V
        compute_dtype = Q.dtype

    O = np.zeros((N, d), dtype=compute_dtype)

    for i_start in range(0, N, Br):
        i_end = min(i_start + Br, N)

        Qi = Qc[i_start:i_end, :]

        # Initialize running output, row-wise max, and row-wise normalizer
        Oi = np.zeros((i_end - i_start, d), dtype=compute_dtype)
        mi = np.full((i_end - i_start,), -np.inf, dtype=compute_dtype)
        li = np.zeros((i_end - i_start,), dtype=compute_dtype)

        for j_start in range(0, N, Bc):
            j_end = min(j_start + Bc, N)

            Kj = Kc[j_start:j_end, :]
            Vj = Vc[j_start:j_end, :]

            # Score block
            Sij = Qi @ Kj.T

            # Local row-wise max and local unnormalized probabilities
            mij = np.max(Sij, axis=1)
            Pij_tilde = np.exp(Sij - mij[:, None])
            lij = np.sum(Pij_tilde, axis=1)

            # Online softmax update
            mi_new = np.maximum(mi, mij)

            li_new = (
                np.exp(mi - mi_new) * li
                + np.exp(mij - mi_new) * lij
            )

            Oi_new = (
                np.exp(mi - mi_new)[:, None] * li[:, None] * Oi
                + np.exp(mij - mi_new)[:, None] * (Pij_tilde @ Vj)
            ) / li_new[:, None]

            mi = mi_new
            li = li_new
            Oi = Oi_new

        O[i_start:i_end, :] = Oi

    return O


# =========================
# Generate Q, K, V
# =========================
def generate_qkv(full_name, precision, N=1024, d=8):
    md5_hash = hashlib.md5(full_name.encode()).hexdigest()
    last_four_hex = md5_hash[-4:]
    seed = int(last_four_hex, 16)

    np.random.seed(seed)

    Q_base = np.random.randn(N, d)
    K_base = np.random.randn(N, d)
    V_base = np.random.randn(N, d)

    Q = cast_precision(Q_base, precision)
    K = cast_precision(K_base, precision)
    V = cast_precision(V_base, precision)

    return Q, K, V, md5_hash, last_four_hex, seed


# =========================
# Run all precisions
# =========================
full_name = "Lin Jiang"
N = 1024
d = 8
M = 256

precisions = ["fp64", "fp32", "fp16", "bf16"]

tolerances = {
    "fp64": (1e-10, 1e-10),
    "fp32": (1e-5, 1e-5),
    "fp16": (1e-2, 1e-2),
    "bf16": (5e-2, 5e-2),
}

for precision in precisions:
    print("=" * 60)
    print("Precision:", precision)

    Q, K, V, md5_hash, last_four_hex, seed = generate_qkv(
        full_name, precision, N=N, d=d
    )

    print("Full name:", full_name)
    print("MD5 hash:", md5_hash)
    print("Last four hex digits:", last_four_hex)
    print("Seed:", seed)
    print("Q shape:", Q.shape)
    print("K shape:", K.shape)
    print("V shape:", V.shape)

    O_ref = standard_attention(Q, K, V, precision)
    O_fa1 = flash_attention_1_forward(Q, K, V, M=M, precision=precision)

    max_diff = np.max(np.abs(O_fa1 - O_ref))

    atol, rtol = tolerances[precision]
    match = np.allclose(O_fa1, O_ref, atol=atol, rtol=rtol)

    print("O_ref shape:", O_ref.shape)
    print("O_fa1 shape:", O_fa1.shape)
    print("Maximum absolute difference:", max_diff)
    print(f"Tolerance: atol={atol}, rtol={rtol}")
    print("Match:", match)

Precision: fp64
Full name: Lin Jiang
MD5 hash: 6d826dfe5d3769ec4ace6aa301a11f80
Last four hex digits: 1f80
Seed: 8064
Q shape: (1024, 8)
K shape: (1024, 8)
V shape: (1024, 8)
O_ref shape: (1024, 8)
O_fa1 shape: (1024, 8)
Maximum absolute difference: 6.661338147750939e-15
Tolerance: atol=1e-10, rtol=1e-10
Match: True
Precision: fp32
Full name: Lin Jiang
MD5 hash: 6d826dfe5d3769ec4ace6aa301a11f80
Last four hex digits: 1f80
Seed: 8064
Q shape: (1024, 8)
K shape: (1024, 8)
V shape: (1024, 8)
O_ref shape: (1024, 8)
O_fa1 shape: (1024, 8)
Maximum absolute difference: 2.1457672e-06
Tolerance: atol=1e-05, rtol=1e-05
Match: True
Precision: fp16
Full name: Lin Jiang
MD5 hash: 6d826dfe5d3769ec4ace6aa301a11f80
Last four hex digits: 1f80
Seed: 8064
Q shape: (1024, 8)
K shape: (1024, 8)
V shape: (1024, 8)
O_ref shape: (1024, 8)
O_fa1 shape: (1024, 8)
Maximum absolute difference: 2.3841858e-06
Tolerance: atol=0.01, rtol=0.01
Match: True
Precision: bf16
Full name: Lin Jiang
MD5 hash: 6d826dfe5d3769ec4

In [6]:
import numpy as np
import pandas as pd
from collections import Counter

# =========================
# Problem settings
# =========================

vocab = [
    "Florida",
    "football",
    "games",
    "hard",
    "play",
    "Seminoles",
    "State",
    "they",
    "win",
]

# Target distribution p(x) from the table
p = np.array([
    0.005928615,
    0.013629966,
    0.018906115,
    0.224970352,
    0.133605673,
    0.000175427,
    0.263495060,
    0.283198600,
    0.055614870,
], dtype=np.float64)

# Normalize p in case printed probabilities have small rounding error
p = p / p.sum()

text = """
Florida State Seminoles play hard football they win games they play hard they win Florida State win football
games they play play win Seminoles play hard football they win games Florida State Seminoles play hard
Florida State play football they win hard games Seminoles play football Seminoles win games they play hard
Florida State football they win games they play games they win Florida State Seminoles play football games
win they win they play hard football Seminoles win Florida State games they play win games Florida State
Seminoles play hard win football games football games they play hard they win win win Florida State
Seminoles play games they play football Seminoles play hard Seminoles win hard they play Florida State
football win games games they win games they play hard Florida State Seminoles play football they win games
they play hard football Florida State Seminoles win win they play games they win football games Seminoles
play hard Florida State Seminoles win hard football games play they play they win hard Florida State
Seminoles win football games
"""

tokens = text.split()
counts = Counter(tokens)
total_tokens = len(tokens)

q = np.array([counts[word] / total_tokens for word in vocab], dtype=np.float64)
q = q / q.sum()

print("Total tokens:", total_tokens)

q_table = pd.DataFrame({
    "Word": vocab,
    "Count in text": [counts[word] for word in vocab],
    "p(x)": p,
    "q(x)": q,
    "Expected count 10000*p(x)": 10000 * p,
    "Expected accepted from q 10000*min(p,q)": 10000 * np.minimum(p, q),
})

print("\nUnigram q(x) table:")
print(q_table.to_string(index=False))


# =========================
# Helper functions
# =========================

def summarize_runs(run_counts, vocab):
    """
    run_counts: shape (num_runs, vocab_size)
    """
    return pd.DataFrame({
        "Word": vocab,
        "Mean": run_counts.mean(axis=0),
        "Minimum": run_counts.min(axis=0),
        "Maximum": run_counts.max(axis=0),
    })


def direct_sampling(p, num_samples=10000, num_runs=10, seed=0):
    rng = np.random.default_rng(seed)
    all_counts = []

    for _ in range(num_runs):
        samples = rng.choice(len(p), size=num_samples, p=p)
        counts = np.bincount(samples, minlength=len(p))
        all_counts.append(counts)

    return np.array(all_counts)


def speculative_sampling(p, q, num_samples=10000, num_runs=10, seed=0):
    """
    Speculative sampling for one-token categorical sampling.

    Step:
    1. sample proposal x from q
    2. accept with probability min(1, p(x)/q(x))
    3. if rejected, sample from residual distribution:
       r(x) proportional to max(p(x)-q(x), 0)
    """

    rng = np.random.default_rng(seed)

    accept_prob = np.minimum(1.0, p / q)

    residual = np.maximum(p - q, 0.0)
    residual_sum = residual.sum()

    if residual_sum > 0:
        residual = residual / residual_sum
    else:
        residual = np.ones_like(p) / len(p)

    all_total_counts = []
    all_accepted_counts = []

    for _ in range(num_runs):
        total_counts = np.zeros(len(p), dtype=int)
        accepted_counts = np.zeros(len(p), dtype=int)

        for _ in range(num_samples):
            proposal = rng.choice(len(p), p=q)
            u = rng.random()

            if u <= accept_prob[proposal]:
                # Accepted from q
                output = proposal
                accepted_counts[output] += 1
            else:
                # Rejected, sample from residual distribution
                output = rng.choice(len(p), p=residual)

            total_counts[output] += 1

        all_total_counts.append(total_counts)
        all_accepted_counts.append(accepted_counts)

    return np.array(all_total_counts), np.array(all_accepted_counts)


# =========================
# 1. Direct sampling from p(x)
# =========================

direct_counts = direct_sampling(
    p=p,
    num_samples=10000,
    num_runs=10,
    seed=123,
)

direct_summary = summarize_runs(direct_counts, vocab)

print("\nDirect sampling from p(x):")
print(direct_summary.to_string(index=False))


# =========================
# 2. Speculative sampling using p(x) and q(x)
# =========================

spec_total_counts, spec_accepted_counts = speculative_sampling(
    p=p,
    q=q,
    num_samples=10000,
    num_runs=10,
    seed=456,
)

spec_total_summary = summarize_runs(spec_total_counts, vocab)
spec_accepted_summary = pd.DataFrame({
    "Word": vocab,
    "Accepted-from-q Mean": spec_accepted_counts.mean(axis=0),
    "Accepted-from-q Minimum": spec_accepted_counts.min(axis=0),
    "Accepted-from-q Maximum": spec_accepted_counts.max(axis=0),
})

print("\nSpeculative sampling total output counts:")
print(spec_total_summary.to_string(index=False))

print("\nSpeculative sampling accepted-from-q counts:")
print(spec_accepted_summary.to_string(index=False))


# =========================
# 3. Comparison
# =========================

comparison = pd.DataFrame({
    "Word": vocab,
    "Expected 10000*p(x)": 10000 * p,
    "Direct mean": direct_counts.mean(axis=0),
    "Speculative total mean": spec_total_counts.mean(axis=0),
    "Expected accepted 10000*min(p,q)": 10000 * np.minimum(p, q),
    "Accepted-from-q mean": spec_accepted_counts.mean(axis=0),
})

print("\nComparison:")
print(comparison.to_string(index=False))


# Optional: overall acceptance rate
acceptance_rate = spec_accepted_counts.sum(axis=1) / 10000
print("\nAcceptance rate over 10 runs:")
print("Mean:", acceptance_rate.mean())
print("Minimum:", acceptance_rate.min())
print("Maximum:", acceptance_rate.max())

Total tokens: 176

Unigram q(x) table:
     Word  Count in text     p(x)     q(x)  Expected count 10000*p(x)  Expected accepted from q 10000*min(p,q)
  Florida             14 0.005931 0.079545                  59.314343                                59.314343
 football             17 0.013636 0.096591                 136.364477                               136.364477
    games             21 0.018915 0.119318                 189.151058                               189.151058
     hard             16 0.225077 0.090909                2250.773362                               909.090909
     play             26 0.133669 0.147727                1336.692089                              1336.692089
Seminoles             16 0.000176 0.090909                   1.755104                                 1.755104
    State             14 0.263620 0.079545                2636.203646                               795.454545
     they             25 0.283333 0.142045                2833.332745    